# Phase 2 · Experiment 2B — Position vs Content

### Google Colab notebook (independent)

Research question: **is the attention sink determined by token position or token content?** We manipulate **only the first token** (keeping the rest of the prompt byte-identical) across five conditions and measure the `mean_from_k` sink score. Conditions are resolved **tokenizer-agnostically** — each candidate is verified to encode as exactly one token, with automatic fallback. Hypothesis: if the sink is position-driven, all conditions give similar scores.

**Runtime:** GPU (free **T4** is enough).

**Prerequisites**
- Phase 1 must have been run with `USE_DRIVE=True`, so `attention_sink_data/` is in your Drive project folder.
- `phase2_utils.py` must be uploaded into the same Drive project folder (upload once; it persists).
- Downloads Qwen3-1.7B for fresh extraction of the controlled prompts.

This notebook is self-contained: it can be rerun on its own without executing the other experiments.

---

## 0. Setup

In [ ]:
# --- Colab environment setup -------------------------------------------------
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *['transformers>=4.51', 'datasets>=2.19', 'accelerate', 'scipy', 'pyarrow']], check=True)

print('In Colab:', IN_COLAB)
import torch
print('CUDA:', torch.cuda.is_available(),
      ('| ' + torch.cuda.get_device_name(0)) if torch.cuda.is_available() else '')
if torch.cuda.is_available():
    print('bf16 native:', torch.cuda.is_bf16_supported(), '(T4=False -> fp16 used)')
else:
    print('*** No GPU. Runtime > Change runtime type > GPU (T4). ***')


In [ ]:
# --- Storage + phase2_utils bootstrap ---------------------------------------
# Point at the SAME Drive project folder Phase 1 used, so Phase 1's raw
# attention and this project's phase2_utils.py are both visible.
USE_DRIVE = True   # must match the Phase 1 setting

import sys
from pathlib import Path
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/attention_sink_project')
elif IN_COLAB:
    BASE = Path('/content/attention_sink_project')
else:
    BASE = Path('.')
BASE.mkdir(parents=True, exist_ok=True)

DATA_ROOT    = str(BASE / 'attention_sink_data')          # Phase 1 output
RESULTS_ROOT = str(BASE / 'results' / 'phase2')           # Phase 2 output
Path(RESULTS_ROOT).mkdir(parents=True, exist_ok=True)

# Locate phase2_utils.py (upload it into BASE once; it persists on Drive).
for c in [BASE, Path('/content'), Path('.')]:
    if (Path(c) / 'phase2_utils.py').exists():
        sys.path.insert(0, str(c)); break
try:
    import phase2_utils as U
    print('phase2_utils loaded from', U.__file__)
except ModuleNotFoundError:
    raise SystemExit('Place phase2_utils.py in ' + str(BASE) + ' (or /content) and re-run this cell.')

print('Phase 1 data :', DATA_ROOT)
print('Phase 2 out  :', RESULTS_ROOT)


## 1. Configuration

In [ ]:
from dataclasses import dataclass, asdict
import json, numpy as np, pandas as pd

@dataclass
class Config2B:
    model_name: str = 'Qwen/Qwen3-1.7B'
    prompt_mode: str = 'prepend_bos'   # matches Phase 1 token-0 standardisation
    k: int = 4
    n_bodies: int = 10                 # number of fixed prompt bodies to average over
    seed: int = 20240517

cfg = Config2B()
U.set_reproducibility(cfg.seed)
EXP = U.experiment_dir(RESULTS_ROOT, 'experiment2B')
logger = U.get_logger('2B', log_file=str(EXP / 'run.log'))
logger.info('Config: %s', asdict(cfg))

## 2. Load model & resolve first-token conditions (auto single-token check)

In [ ]:
tokenizer, model = U.load_model(cfg.model_name, dtype='auto', logger=logger)
prefix_id = U.choose_prefix_token_id(tokenizer)
conditions = U.resolve_2b_conditions(tokenizer, prefix_id, logger=logger)
with open(EXP / 'conditions.json', 'w') as f:
    json.dump(conditions, f, indent=2)
conditions

## 3. Fixed prompt bodies (identical across all conditions)

In [ ]:
try:
    m1 = pd.read_csv(Path(DATA_ROOT) / 'metadata.csv')
    bodies = m1.loc[m1['language'] == 'en', 'text'].dropna().tolist()[:cfg.n_bodies]
except Exception:
    bodies = []
if len(bodies) < cfg.n_bodies:
    bodies = [p['eng'] for p in U.EMBEDDED_PAIRS] * 3
    bodies = bodies[:cfg.n_bodies]
body_ids = [U.body_ids_from_text(tokenizer, b) for b in bodies]
print('using', len(body_ids), 'bodies; token lengths:', [len(b) for b in body_ids])

## 4. Extract sink matrix per (body, condition)

In [ ]:
records = []                 # long-form rows
per_cond = {c: [] for c in conditions}   # condition -> list of [L,H] over bodies
for bi, bids in enumerate(body_ids):
    for cname, cinfo in conditions.items():
        ids = U.build_first_token_variant(cinfo['token_id'], bids)
        M = U.extract_sink_matrix_for_ids(ids, model, k=cfg.k)   # [L,H]
        per_cond[cname].append(M)
        L, H = M.shape
        for l in range(L):
            for h in range(H):
                records.append({'body': bi, 'condition': cname, 'layer': l, 'head': h,
                                'sink_score': float(M[l, h])})
    logger.info('body %d/%d done', bi + 1, len(body_ids))
per_cond = {c: np.stack(v) for c, v in per_cond.items()}     # each [n_bodies, L, H]
df = pd.DataFrame(records)
df.to_csv(EXP / 'sink_by_condition.csv', index=False)
print('records:', df.shape)

## 5. Statistics & effect sizes

In [ ]:
# per-body global sink score for each condition (paired across bodies)
cond_glob = {c: per_cond[c].mean(axis=(1, 2)) for c in conditions}   # [n_bodies]
cond_global_mean = {c: float(v.mean()) for c, v in cond_glob.items()}
cond_ci = {c: U.bootstrap_ci(v) for c, v in cond_glob.items()}

# omnibus effect of first-token identity across the 5 conditions
eta2 = U.eta_squared_oneway(list(cond_glob.values()))

# each content condition vs the BOS/position baseline (paired)
vs_bos = {}
for c in conditions:
    if c == 'BOS':
        continue
    vs_bos[c] = U.paired_test(cond_glob[c], cond_glob['BOS'])

stats = {'condition_global_mean': cond_global_mean, 'condition_ci95': cond_ci,
         'omnibus_eta_squared': eta2, 'vs_BOS_paired': vs_bos,
         'global_spread': float(max(cond_global_mean.values()) - min(cond_global_mean.values()))}
with open(EXP / 'stats.json', 'w') as f:
    json.dump(stats, f, indent=2)
pd.DataFrame([{'condition': c, 'global_mean': cond_global_mean[c],
               'ci_lo': cond_ci[c][0], 'ci_hi': cond_ci[c][1]} for c in conditions]
             ).to_csv(EXP / 'condition_effect_sizes.csv', index=False)
print('omnibus eta^2 (first-token identity):', round(eta2, 4))
print('global spread across conditions      :', round(stats['global_spread'], 4))
for c, t in vs_bos.items():
    print('  %-13s vs BOS: paired d=%+.3f' % (c, t['cohens_d_paired']))

## 6. Comparison plots + hypothesis read-out

In [ ]:
cond_layer_prof = {c: per_cond[c].mean(axis=(0, 2)) for c in conditions}   # [L] each
U.plot_condition_global_bar(cond_global_mean, cond_ci, EXP / 'figures', 'condition_global_bar.png')
U.plot_condition_layer_profiles(cond_layer_prof, EXP / 'figures', 'condition_layer_profiles.png')

spread = stats['global_spread']; mean_lvl = np.mean(list(cond_global_mean.values()))
rel = spread / mean_lvl if mean_lvl else float('nan')
print('Hypothesis check (position vs content):')
print('  relative spread across conditions = %.1f%% of mean sink level' % (100 * rel))
print('  eta^2 = %.3f  ->' % eta2,
      'first-token CONTENT matters' if eta2 > 0.06 else 'largely POSITION-driven (conditions similar)')

## Download results

In [ ]:
# --- Download this experiment's outputs -------------------------------------
import shutil
exp = Path(RESULTS_ROOT) / 'experiment2B'
zp = shutil.make_archive(str(Path('/content' if IN_COLAB else '.') / ('experiment2B_outputs')), 'zip', exp)
print('Bundled:', zp)
if IN_COLAB:
    from google.colab import files
    files.download(zp)
